# 03 — Model Training: Classical Anomaly Detection Models

Trains all classical anomaly detection models on the labeled Kepler dataset and exports trained model artifacts (.pkl files).

**Input:** `data/labeled/` from notebook 01
**Output:** `models/*.pkl` — trained model files for local Streamlit use

In [ ]:
!pip install -q lightkurve astroquery PyWavelets matplotlib

import sys, os, glob

# Auto-detect codebase path
CODEBASE_PATH = None
for candidate in [
    '/kaggle/input/exopattern-codebase',
    *glob.glob('/kaggle/input/exopattern-codebase/*/'),
    *glob.glob('/kaggle/input/exopattern*/'),
    *glob.glob('/kaggle/input/exopattern*/*/'),
]:
    if os.path.isdir(os.path.join(candidate, 'backend')):
        CODEBASE_PATH = candidate.rstrip('/')
        break

if CODEBASE_PATH is None:
    raise FileNotFoundError(
        "Could not find 'backend/' directory in /kaggle/input/. "
        "Upload the project root (containing backend/ and experiments/) as a Kaggle dataset."
    )

sys.path.insert(0, CODEBASE_PATH)
print(f"Codebase found at: {CODEBASE_PATH}")

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path

from backend.ml.preprocessing import LightCurvePreprocessor
from backend.ml.model_registry import get_model, list_models, get_display_name
from backend.ml.feature_names import get_feature_names

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load Dataset and Extract Features

In [ ]:
# Auto-detect data directory
DATA_DIR = None
for candidate in [
    '/kaggle/input/exopattern-labeled-data/data/labeled',
    *glob.glob('/kaggle/input/exopattern-labeled*/data/labeled'),
    *glob.glob('/kaggle/input/exopattern-labeled*/*/data/labeled'),
    '/kaggle/working/data/labeled',
    'data/labeled',
]:
    if os.path.exists(os.path.join(candidate, 'metadata.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find labeled dataset (metadata.csv) in /kaggle/input/")

print(f"Using data from: {DATA_DIR}")

metadata = pd.read_csv(f'{DATA_DIR}/metadata.csv')
print(f"Loaded metadata for {len(metadata)} targets")

pp = LightCurvePreprocessor()
window_size = 50

all_features = []
all_labels = []

for _, row in metadata.iterrows():
    lc_path = f"{DATA_DIR}/lightcurves/{row['filename']}"
    if not os.path.exists(lc_path):
        continue
    df = pd.read_csv(lc_path)
    labels = df['label'].values
    df_proc = pp.preprocess(df, normalize=True)
    features = pp.extract_features(df_proc, window_size)
    
    stride = max(1, window_size // 4)
    window_labels = []
    for i in range(0, len(df_proc) - window_size + 1, stride):
        window_lab = labels[i:i + window_size]
        window_labels.append(int(window_lab.max() > 0))
    
    n = min(len(features), len(window_labels))
    all_features.append(features[:n])
    all_labels.append(np.array(window_labels[:n]))

X = np.vstack(all_features)
y = np.concatenate(all_labels)

# Replace NaN/inf with 0
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Dataset: {X.shape[0]} windows, {X.shape[1]} features")
print(f"Anomalous: {y.sum()} ({y.mean()*100:.1f}%)")

## 2. Train All Classical Models

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classical_models = ['isolation_forest', 'lof', 'ocsvm', 'dbscan', 'ensemble']
trained_models = {}

for model_name in classical_models:
    print(f"\nTraining: {get_display_name(model_name)}")
    model = get_model(model_name, contamination=0.1, random_state=RANDOM_SEED)
    model.fit(X)
    
    # Quick evaluation on training data
    predictions = model.predict(X)
    pred_anomaly = (predictions == -1) if predictions.min() < 0 else predictions.astype(bool)
    
    from sklearn.metrics import precision_score, recall_score, f1_score
    p = precision_score(y, pred_anomaly, zero_division=0)
    r = recall_score(y, pred_anomaly, zero_division=0)
    f1 = f1_score(y, pred_anomaly, zero_division=0)
    
    print(f"  Train P={p:.3f} R={r:.3f} F1={f1:.3f}")
    
    # Save model
    model_path = OUTPUT_DIR / f"{model_name}.pkl"
    joblib.dump(model, model_path)
    print(f"  Saved to {model_path}")
    
    trained_models[model_name] = model

print(f"\nAll {len(trained_models)} models trained and saved.")

## 3. Also Save the Preprocessor Config

In [ ]:
# Save training metadata for reproducibility
training_meta = {
    'n_targets': len(metadata),
    'n_windows': int(X.shape[0]),
    'n_features': int(X.shape[1]),
    'n_anomalous': int(y.sum()),
    'anomaly_fraction': float(y.mean()),
    'window_size': window_size,
    'feature_groups': ['statistical', 'frequency', 'wavelet', 'autocorrelation', 'shape'],
    'contamination': 0.1,
    'random_seed': RANDOM_SEED,
    'models_trained': classical_models,
}

with open(OUTPUT_DIR / 'training_metadata.json', 'w') as f:
    json.dump(training_meta, f, indent=2)

print(json.dumps(training_meta, indent=2))

## 4. Verify Saved Models Load Correctly

In [ ]:
# Verify all models load and predict
for model_name in classical_models:
    loaded = joblib.load(OUTPUT_DIR / f"{model_name}.pkl")
    preds = loaded.predict(X[:10])
    print(f"{model_name}: loaded OK, sample predictions = {preds}")

print("\nAll models verified.")

In [ ]:
# List output files
print("Output files:")
for f in sorted(OUTPUT_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")

## Done!

Download the `models/` directory from this notebook's output. Place in `artifacts/models/` locally for Streamlit to use.

Next: Run notebook 04 (experiments) or 05 (deep learning).

## 5. Package Models as ZIP

In [ ]:
import shutil

# Zip all trained classical models (.pkl + metadata)
shutil.make_archive('/kaggle/working/exopattern-classical-models', 'zip', '/kaggle/working/models')
zip_size = os.path.getsize('/kaggle/working/exopattern-classical-models.zip') / (1024 * 1024)
print(f"Created exopattern-classical-models.zip ({zip_size:.1f} MB)")
print("Download and place contents in artifacts/models/ locally.")